# Multiple Lineare Regression

**Ziel:** Sie trainieren ein Regressionsmodell für **beliebige CSV-Datensätze mit numerischer Zielvariable**. Sie müssen den Python-Code nicht vollständig verstehen: Passen Sie nur die **Konfigurationszelle** (Schritt 3) an, führen Sie die Codezellen von oben nach unten aus und interpretieren Sie die Ergebnisse. Alle anderen Code-Zellen sind ausgeblendet – mit einem Doppelklick auf eine Zelle können Sie den Code bei Interesse einblenden.

**Ablauf:** CSV hochladen → Daten ansehen → Features und Zielvariable auswählen → Zusammenhänge ansehen → Trainieren → Regressionsfunktion und Fehler ausgeben → Istwerte und Vorhersagen vergleichen.

**Beispiele für die Konfiguration:**
- Advertising: `FEATURES = ['TV', 'radio', 'newspaper']`, `TARGET = 'sales'`
- Fußballspieler: `FEATURES = ['goalsTotal', 'age']`, `TARGET = 'marketvalue'` (nur, wenn diese Spalten tatsächlich im CSV vorkommen).

**Hinweis:** Dieses Notebook verwendet eine **multiple lineare Regression** (ohne polynomiale Terme oder kategoriales Encoding). Die Eingabevariablen müssen numerisch sein. Sie können auch nur ein Feature wählen.

**Modell:** $\hat y = \hat\beta_0 + \hat\beta_1 x_1 + \hat\beta_2 x_2 + \dots + \hat\beta_p x_p$

## 1. CSV-Datei hochladen
**Was passiert hier?** Es öffnet sich ein Dialog, über den Sie eine CSV-Datei von Ihrem Computer in die Colab-Sitzung hochladen (z. B. `Advertising.csv` oder `players.csv`). Die Datei steht anschließend für alle weiteren Schritte zur Verfügung.

In [ ]:
# @title 1. CSV-Datei hochladen { display-mode: "form" }
# Colab-Funktion zum Hochladen von Dateien importieren
from google.colab import files

# Upload-Dialog öffnen; Ergebnis ist ein Dictionary {Dateiname: Dateiinhalt (Bytes)}
uploaded = files.upload()

# Abbrechen mit verständlicher Meldung, falls keine Datei ausgewählt wurde
if not uploaded:
    raise ValueError('Bitte eine CSV-Datei auswählen.')

## 2. Daten ansehen
**Was passiert hier?** Die hochgeladene CSV-Datei wird als Tabelle (DataFrame) eingelesen. Angezeigt werden die ersten fünf Zeilen, damit Sie die **Spaltennamen** sehen – diese benötigen Sie im nächsten Schritt für die Konfiguration.

In [ ]:
# @title 2. Daten einlesen und anzeigen { display-mode: "form" }
import io
import pandas as pd
import numpy as np

# Name der hochgeladenen Datei ermitteln und Inhalt als DataFrame einlesen
filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Eventuell mitexportierte Indexspalten ("Unnamed: 0") entfernen
df = df.loc[:, ~df.columns.astype(str).str.match(r'^Unnamed(:|$)')]

# Die ersten fünf Zeilen anzeigen
display(df.head())

## 3. Konfiguration – nur hier anpassen
**Was passiert hier?** Sie legen fest, welche Spalten als Eingabevariablen (`FEATURES`) und welche Spalte als Zielvariable (`TARGET`) verwendet werden. Wählen Sie aus den oben angezeigten Spalten **eine oder mehrere numerische Eingabevariablen** und **eine numerische Zielvariable**. Schreibweise und Groß-/Kleinschreibung müssen exakt stimmen.

Zusätzlich legen Sie fest, welcher Anteil der Daten als Testdaten zurückgehalten wird (`TEST_SIZE`). Nach einer Änderung **alle folgenden Zellen erneut ausführen**.

In [ ]:
# ===== KONFIGURATION: NUR HIER ANPASSEN =====
FEATURES = ['TV', 'radio', 'newspaper']   # Eingabevariablen – alternativ z. B. ['goalsTotal', 'age']
TARGET = 'sales'                          # Zielvariable     – alternativ z. B. 'marketvalue'
TEST_SIZE = 0.20                          # Anteil der Testdaten (0.20 = 20 %)
# ==========================================

**Konfiguration prüfen:** Die folgende Zelle kontrolliert Ihre Eingaben (Spalten vorhanden, numerisch, keine Doppelungen, Zielvariable nicht zugleich Feature) und entfernt Zeilen mit fehlenden Werten. Bei einem Fehler erhalten Sie einen Hinweis, was Sie in der Konfiguration korrigieren müssen.

In [ ]:
# @title 3b. Konfiguration prüfen und Daten vorbereiten { display-mode: "form" }
# Fester Startwert für die zufällige Aufteilung, damit alle dieselbe Aufteilung erhalten
RANDOM_STATE = 42

# FEATURES muss eine nicht-leere Liste sein
if not isinstance(FEATURES, list) or not FEATURES:
    raise ValueError("FEATURES muss eine nicht-leere Liste sein, z. B. ['TV', 'radio'].")

# Kein Feature darf doppelt vorkommen
if len(FEATURES) != len(set(FEATURES)):
    raise ValueError('Bitte kein Feature mehrfach auswählen.')

# Die Zielvariable darf nicht gleichzeitig Eingabevariable sein
if TARGET in FEATURES:
    raise ValueError('Die Zielvariable darf nicht zugleich ein Feature sein.')

# Prüfen, ob alle Spalten im Datensatz existieren
missing = [col for col in FEATURES + [TARGET] if col not in df.columns]
if missing:
    raise ValueError(f'Spalten nicht gefunden: {missing}. Vorhanden: {list(df.columns)}')

# Prüfen, ob alle Spalten numerisch sind (Regression braucht Zahlen)
non_numeric = [col for col in FEATURES + [TARGET] if not pd.api.types.is_numeric_dtype(df[col])]
if non_numeric:
    raise ValueError(f'Diese Spalten sind nicht numerisch: {non_numeric}. Wählen Sie numerische Spalten.')

# Nur die benötigten Spalten behalten; unendliche und fehlende Werte entfernen
model_df = df[FEATURES + [TARGET]].replace([np.inf, -np.inf], np.nan).dropna().copy()
if len(model_df) < 10:
    raise ValueError('Zu wenige vollständige Beobachtungen (mindestens 10 erforderlich).')

# Zusammenfassung ausgeben
print('Features:', FEATURES)
print('Zielvariable:', TARGET)
print('Verwendbare Beobachtungen:', len(model_df), '(entfernte unvollständige Zeilen:', len(df)-len(model_df), ')')

## 4. Zusammenhänge erkunden
**Was passiert hier?** Für jedes ausgewählte Feature wird ein eigener Scatterplot gegen die Zielvariable gezeichnet. Jedes Diagramm zeigt **jeweils nur einen Zusammenhang** (ein Feature und die Zielvariable); die multiple Regression betrachtet dagegen alle ausgewählten Features gemeinsam.

In [ ]:
# @title 4. Scatterplots { display-mode: "form" }
import matplotlib.pyplot as plt

# Ein Diagramm pro Feature nebeneinander anlegen
fig, axes = plt.subplots(1, len(FEATURES), figsize=(5 * len(FEATURES), 4), squeeze=False)

# Für jedes Feature: Feature (x-Achse) gegen Zielvariable (y-Achse) zeichnen
for ax, feature in zip(axes.ravel(), FEATURES):
    ax.scatter(model_df[feature], model_df[TARGET], alpha=0.65)
    ax.set_xlabel(feature)
    ax.set_ylabel(TARGET)
    ax.set_title(f'{TARGET} vs. {feature}')
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 5. Trainings- und Testdaten aufteilen
**Was passiert hier?** Die Daten werden zufällig in **Trainingsdaten** und **Testdaten** aufgeteilt (standardmäßig 80 % / 20 %, gemäß `TEST_SIZE`). Das Modell wird ausschließlich mit den Trainingsdaten geschätzt; die Testdaten dienen dazu, Vorhersagen auf bislang ungesehenen Beobachtungen zu bewerten.

In [ ]:
# @title 5. Trainings- und Testdaten aufteilen { display-mode: "form" }
from sklearn.model_selection import train_test_split

# Eingabevariablen X (Tabelle mit allen Features) und Zielvariable y
X = model_df[FEATURES]
y = model_df[TARGET]

# Zufällige Aufteilung in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# Größe der beiden Teilmengen ausgeben
print('Training:', len(X_train), '| Test:', len(X_test))

## 6. Multiple lineare Regression trainieren
**Was passiert hier?** Das Modell wird mit den Trainingsdaten angepasst. Der Algorithmus bestimmt den Achsenabschnitt (Intercept) $\hat\beta_0$ und für jedes Feature einen Koeffizienten $\hat\beta_j$, indem er den **Trainings-RSS** (Summe der quadrierten Abweichungen) minimiert. Ausgegeben werden die Koeffizienten und die gelernte Regressionsfunktion.

In [ ]:
# @title 6. Modell trainieren und Regressionsfunktion anzeigen { display-mode: "form" }
from sklearn.linear_model import LinearRegression

# Lineares Regressionsmodell erstellen und mit den Trainingsdaten anpassen
model = LinearRegression()
model.fit(X_train, y_train)

# Achsenabschnitt und Koeffizienten (einer pro Feature) als Tabelle ausgeben
print(f'Intercept: {model.intercept_:.5f}')
print('Koeffizienten:')
display(pd.DataFrame({'Feature': FEATURES, 'Koeffizient': model.coef_}).round(5))

# Regressionsfunktion als Text zusammensetzen: Intercept + Summe(Koeffizient × Feature)
terms = [f'({coef:+.5f} × {feature})' for feature, coef in zip(FEATURES, model.coef_)]
print('Geschätzte Regressionsfunktion:')
print(f'{TARGET}_vorhergesagt = {model.intercept_:.5f} ' + ' '.join(terms))

## 7. Vorhersagen für Testdaten und Modellfehler
**Was passiert hier?** Das trainierte Modell sagt die Zielvariable für die Testdaten vorher. Eine Tabelle zeigt für die ersten zehn Testbeobachtungen Istwert, Prognose und **Residuum** (Ist − Prognose). Anschließend werden Fehlermaße auf den Testdaten berechnet:

**SSE/RSS:** Summe der quadrierten Fehler; **MSE:** mittlerer quadrierter Fehler; **RMSE:** Wurzel aus dem MSE – Fehlermaß in der Einheit der Zielvariable.

In [ ]:
# @title 7. Testvorhersagen und Modellfehler { display-mode: "form" }
from sklearn.metrics import mean_squared_error

# Vorhersagen für die Testdaten berechnen
y_pred = model.predict(X_test)

# Vergleichstabelle: Feature-Werte, Istwert, Prognose und Residuum
resultate = X_test.copy()
resultate['Istwert'] = y_test.to_numpy()
resultate['Prognose'] = y_pred
resultate['Residuum'] = resultate['Istwert'] - resultate['Prognose']
display(resultate.head(10).round(3))

# Fehlermaße auf den Testdaten berechnen
sse = float(np.sum((y_test.to_numpy() - y_pred)**2))   # Summe der quadrierten Residuen
mse = mean_squared_error(y_test, y_pred)               # mittlerer quadrierter Fehler
rmse = np.sqrt(mse)                                    # Wurzel aus MSE (Einheit der Zielvariable)

print(f'SSE/RSS  = {sse:.3f}')
print(f'MSE      = {mse:.3f}')
print(f'RMSE     = {rmse:.3f}')
print('Alle Kennzahlen wurden auf dem TEST-Datensatz berechnet.')

## 8. Tatsächliche und vorhergesagte Zielwerte vergleichen
**Was passiert hier?** Für jede Testbeobachtung wird der tatsächliche Wert (x-Achse) gegen den vorhergesagten Wert (y-Achse) aufgetragen. Je näher die Punkte an der gestrichelten Diagonale liegen, desto genauer stimmen Prognose und Istwert überein.

In [ ]:
# @title 8. Istwerte und Vorhersagen vergleichen { display-mode: "form" }
# Istwerte gegen Vorhersagen der Testdaten zeichnen
plt.figure(figsize=(6,5))
plt.scatter(y_test, y_pred, alpha=0.75)

# Diagonale y = x als Referenz für eine perfekte Prognose
lo = min(y_test.min(), y_pred.min())
hi = max(y_test.max(), y_pred.max())
plt.plot([lo,hi], [lo,hi], linestyle='--', color='black', label='Perfekte Prognose')

# Beschriftungen und Legende
plt.xlabel(f'{TARGET} – tatsächlich')
plt.ylabel(f'{TARGET} – vorhergesagt')
plt.title('Tatsächlich vs. vorhergesagt')
plt.legend()
plt.grid(alpha=0.2)
plt.show()